In [ ]:
import os
import time
import random
import re
import json
import urllib.parse
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper
import sys
import warnings

# 🌟 FIX 1: Suppress Pandas FutureWarnings to keep the console clean
warnings.simplefilter(action='ignore', category=FutureWarning)

print("🚀 Starting the Autonomous TTRPG Feature Engineering Pipeline (HTML-ONLY EDITION)...")

# --- SETUP ---
INPUT_FILE = "ttrpg_database_final.csv"
SCRAPED_FILE = "scraped_rpg_features_part2.csv"
FINAL_OHE_FILE = "ttrpg_features_ohe_final.csv"

# ==========================================
# BLOCK 1: Stratified Sampling (Exactly 500 per Tier)
# ==========================================
print("\n--- BLOCK 1: STRATIFIED SAMPLING DATA ---")
try:
    df_main = pd.read_csv(INPUT_FILE)
    print(f"📦 Loaded main dataset with {len(df_main)} total games.")
except FileNotFoundError:
    print(f"🚨 Error: Could not find '{INPUT_FILE}'. Please ensure it is in the directory.")
    sys.exit()

# If the file already exists from a previous run, load it so we don't start over!
if os.path.exists(SCRAPED_FILE):
    df_sampled = pd.read_csv(SCRAPED_FILE)
    print(f"🔄 Resuming from existing '{SCRAPED_FILE}' with {len(df_sampled)} games.")
else:
    # 🌟 FIX 2: Added 0-1 to the bins and labels
    bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-6', '6-7', '7-8', '8-9', '9-10']
    
    # Categorize each game into a bucket based on its Average Score
    df_main['Score_Tier'] = pd.cut(df_main['Average Score'], bins=bins, labels=labels)
    
    # Sample EXACTLY 500 games from each tier 
    print("Bucketing games by score and sampling exactly 500 per tier...")
    df_sampled = df_main.groupby('Score_Tier', group_keys=False, observed=False).apply(
        lambda x: x.sample(n=500, replace=len(x) < 500, random_state=42)
    ).copy()
    
    print("\n📊 Stratified Distribution:")
    print(df_sampled['Score_Tier'].value_counts().sort_index())
    
    # Clean up the temporary column
    if 'Score_Tier' in df_sampled.columns:
        df_sampled = df_sampled.drop(columns=['Score_Tier'])
    
    # Add empty columns and explicitly cast them as objects (text) to prevent type warnings
    for col in ['rpg_url', 'rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']:
        df_sampled[col] = pd.Series(dtype='object')
        
    # Shuffle the final dataset
    df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)
    
    df_sampled.to_csv(SCRAPED_FILE, index=False)
    print(f"\n🎯 Created stratified sample of {len(df_sampled)} games and saved to '{SCRAPED_FILE}'.")


# ==========================================
# BLOCK 2: Scrape RPGGeek Features (PURE HTML SCRAPING)
# ==========================================
print("\n--- BLOCK 2: WEB SCRAPING FEATURES (HTML ONLY) ---")

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'darwin', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# Find indices of rows that haven't been scraped yet
missing_mask = df_sampled['rpg_url'].isna() & df_sampled['rpgsystem'].isna() & df_sampled['rpggenre'].isna()
missing_indices = df_sampled[missing_mask].index.tolist()

if not missing_indices:
    print("✅ All rows have been processed! Moving to Block 3.")
else:
    print(f"🔍 Found {len(missing_indices)} games needing features scraped.")
    
    session_count = 0
    consecutive_403 = 0
    hard_exit = False
    
    for idx in missing_indices:
        if hard_exit: break
            
        game_name = str(df_sampled.at[idx, 'Name'])
        print(f"[{session_count+1}/{len(missing_indices)}] HTML Scrape: {game_name[:25]:<25}...", end=" ")
        
        try:
            # STEP A: Search the HTML frontend
            encoded_name = urllib.parse.quote(game_name)
            search_url = f"https://boardgamegeek.com/search/rpgitem?q={encoded_name}"
            
            res_search = scraper.get(search_url, headers=human_headers)
            
            # 🌟 CRITICAL DELAY: 4 to 7 seconds to mimic human reading speed.
            time.sleep(4.0) 
            
            if res_search.status_code == 403:
                consecutive_403 += 1
                print(f"| 🚨 403 Blocked by Cloudflare! (Strike {consecutive_403}/3)")
                if consecutive_403 >= 3: hard_exit = True
                continue
                
            consecutive_403 = 0
            soup_search = BeautifulSoup(res_search.content, 'html.parser')
            
            # Find the first game link
            first_link = soup_search.find('a', href=re.compile(r'^/rpgitem/\d+/'))
            
            if not first_link:
                print("| ⏭️ No search results found.")
                df_sampled.at[idx, 'rpg_url'] = "NOT_FOUND"
            else:
                full_url = "https://boardgamegeek.com" + first_link['href']
                df_sampled.at[idx, 'rpg_url'] = full_url
                
                # STEP B: Scrape the actual game page HTML
                res_page = scraper.get(full_url, headers=human_headers)
                # time.sleep(random.uniform(4.0, 7.0)) 
                time.sleep(4.0) 
                
                if res_page.status_code == 403:
                    consecutive_403 += 1
                    print(f"| 🚨 403 on Game Page! (Strike {consecutive_403}/3)")
                    if consecutive_403 >= 3: hard_exit = True
                    continue
                    
                consecutive_403 = 0
                soup_page = BeautifulSoup(res_page.content, 'html.parser')
                
                # Extract the hidden JSON block inside the HTML script tags
                found_data = False
                for script in soup_page.find_all('script'):
                    if script.string and 'GEEK.geekitemPreload' in script.string:
                        json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                        if json_match:
                            try:
                                data = json.loads(json_match.group(1))
                                links = data.get('item', {}).get('links', {})
                                
                                for category in ['rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']:
                                    if category in links and isinstance(links[category], list):
                                        names = [item.get('name') for item in links[category] if item.get('name')]
                                        if names:
                                            df_sampled.at[idx, category] = ", ".join(names)
                                            found_data = True
                            except json.JSONDecodeError:
                                pass
                        break
                        
                if found_data: print("| ✅ HTML parsed & features extracted")
                else: print("| ⚠️ No categories found in HTML")
                
            # Auto-Save every 10 items
            session_count += 1
            if session_count % 10 == 0:
                df_sampled.to_csv(SCRAPED_FILE, index=False)
                print(f"      💾 Auto-Save: Cleaned 10 rows. CSV safely overwritten.")
                
        except Exception as e:
            print(f"| Error: {e}")
            time.sleep(4) 
            
    # Final save for Block 2
    df_sampled.to_csv(SCRAPED_FILE, index=False)
    
    if hard_exit:
        print("\n🚨 SCRAPER STOPPED DUE TO CLOUDFLARE 403 BANS.")
        print("💡 TIP: Turn on a VPN or connect to a mobile hotspot to get a new IP address, then run this cell again.")
        sys.exit("Terminated due to 403 limits.")


# ==========================================
# BLOCK 3: One-Hot Encode All Features
# ==========================================
print("\n--- BLOCK 3: ONE-HOT ENCODING (MACHINE LEARNING PREP) ---")
df_ml = pd.read_csv(SCRAPED_FILE)

cols_to_encode = ['rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']

# Prevent NaN errors by replacing empty cells with empty strings
for col in cols_to_encode:
    df_ml[col] = df_ml[col].fillna('')

def one_hot_encode_column(df, column_name):
    """Splits multi-tags (e.g., 'Fantasy, Horror') and One-Hot Encodes them."""
    s = df[column_name].str.split(', ').apply(lambda x: [item.strip() for item in x if item.strip()])
    
    # Check if the column is entirely empty (no tags extracted at all)
    if s.apply(len).sum() == 0:
        return pd.DataFrame(index=df.index)
        
    ohe_df = s.explode().str.get_dummies().groupby(level=0).sum()
    
    prefix = column_name.replace('rpg', '') + "_"
    ohe_df.columns = [prefix + col for col in ohe_df.columns]
    return ohe_df

print("Generating binary columns...")
ohe_dataframes = [df_ml] 

for col in cols_to_encode:
    print(f"Encoding {col}...")
    ohe_df = one_hot_encode_column(df_ml, col)
    if not ohe_df.empty:
        ohe_dataframes.append(ohe_df)

# Concatenate all the new OHE features alongside the original dataset
df_final = pd.concat(ohe_dataframes, axis=1)

# Drop the original string columns as they are no longer needed
df_final = df_final.drop(columns=cols_to_encode + ['rpg_url'])

df_final.to_csv(FINAL_OHE_FILE, index=False)

print(f"\n🎉 PIPELINE COMPLETE!")
print(f"Original shape: {df_sampled.shape}")
print(f"Final OHE shape: {df_final.shape}")
print(f"Machine Learning ready dataset saved to '{FINAL_OHE_FILE}'.")

🚀 Starting the Autonomous TTRPG Feature Engineering Pipeline (HTML-ONLY EDITION)...

--- BLOCK 1: STRATIFIED SAMPLING DATA ---
📦 Loaded main dataset with 10000 total games.
🔄 Resuming from existing 'scraped_rpg_features_part2.csv' with 2500 games.

--- BLOCK 2: WEB SCRAPING FEATURES (HTML ONLY) ---
🔍 Found 2449 games needing features scraped.
[1/2449] HTML Scrape: The Mechanoids           ... | 🚨 403 Blocked by Cloudflare! (Strike 1/3)
[1/2449] HTML Scrape: The Quintessential Cleric... | 🚨 403 Blocked by Cloudflare! (Strike 2/3)
[1/2449] HTML Scrape: Mayhem at the Truffle Fes... | 🚨 403 Blocked by Cloudflare! (Strike 3/3)

🚨 SCRAPER STOPPED DUE TO CLOUDFLARE 403 BANS.
💡 TIP: Turn on a VPN or connect to a mobile hotspot to get a new IP address, then run this cell again.


SystemExit: Terminated due to 403 limits.

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
!pip install cloudscraper beautifulsoup4 pandas


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
